# Pharmagen — 4. Knowledge graph, training data and evaluation

Training rows come from a **Neo4j knowledge graph** built from PharmGKB,
PharmVar, dbSNP and NCBI. This notebook shows the graph's contents and
provenance, the training rows it exports, the evaluation protocol, and the
metrics from a previous run.

The graph itself needs a running Neo4j, so what the bundle carries — and what
this notebook reads — is the **exported snapshot** plus the audit reports.

In [ ]:
# --- Environment -----------------------------------------------------------
# Colab runs Python 3.13; Pharmagen and Modephere need 3.14. colab_setup lets
# uv provision 3.14 + the locked dependencies and starts a kernel there. Every
# cell below marked %%pgen runs in that 3.14 kernel and keeps its state.
REPO_URL = "https://github.com/Aderfi/Pgen_dev.git"
BRANCH = "modephere-imp"

# The data bundle (scripts/pack_colab_bundle.py, ~190 MB): a Google Drive path,
# or an https URL to a release asset. Ignored when running locally.
BUNDLE = "/content/drive/MyDrive/pharmagen_colab_bundle.zip"
# BUNDLE = "https://github.com/Aderfi/Pgen_dev/releases/download/demo/pharmagen_colab_bundle.zip"

import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO = Path("/content/dev_Pgen")
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
else:
    REPO, BUNDLE = Path.cwd().parent, None  # notebooks/ -> repository root

sys.path.insert(0, str(REPO / "notebooks"))
import colab_setup

kernel = colab_setup.prepare(REPO, BUNDLE)

## How the graph is built

```
  python -m src.kg fetch     the only networked step: dbSNP, NCBI SPDI, ClinPGx
                             -> data/library/kg/raw/   (a content-addressed cache)

  python -m src.kg ingest    offline: raw cache + PharmGKB + PharmVar -> Neo4j
                             every node and relationship gets sources + run_id

  python -m src.kg audit     verifies that provenance; writes reports/kg/

  python -m src.kg export    -> data/library/kg/training_rows.parquet
```

Two rules make the graph trustworthy enough to train on:

1. **Every variant is a canonical NCBI SPDI on GRCh38.** Indel normalization is
   delegated to NCBI rather than reimplemented, and the accession set is
   explicit (GRCh38.p14) rather than inferred from a version number.
2. **Nothing enters without provenance.** Every node and relationship carries
   `sources` and the `run_id` that created it, so any fact can be traced back to
   the dataset it came from.

### The Neo4j property-graph schema and Cypher queries

Even when running offline in Google Colab from the exported snapshot, the full Neo4j schema, uniqueness constraints, and Cypher queries live in `src/kg/graph.py`. If a Neo4j instance is reachable (`docker compose -f docker-compose.kg.yml up -d`), `KnowledgeGraph.connect()` queries it directly; otherwise the notebook reads the exported parquet tables.

In [ ]:
%%pgen
from src.kg.graph import (
    AUDIT,
    NODE_KEYS,
    SCHEMA_REL_TYPES,
    SCHEMA_STATEMENTS,
    TRAINING_ROWS,
    KnowledgeGraph,
)

print("Neo4j Node Labels -> Primary Key:")
for label, key in NODE_KEYS.items():
    print(f"  (:{label:11} {{{key}: ...}})")

print(f"\nNeo4j Relationship Types ({len(SCHEMA_REL_TYPES)}):")
print(" ", ", ".join(f"[:{r}]" for r in SCHEMA_REL_TYPES))

print(f"\nSchema Constraints & Indexes ({len(SCHEMA_STATEMENTS)} statements):")
for stmt in SCHEMA_STATEMENTS[:6]:
    print(" ", stmt)
print(f"  ... ({len(SCHEMA_STATEMENTS) - 6} more index statements)")

print("\nCypher query exporting training_rows.parquet (TRAINING_ROWS):")
for line in TRAINING_ROWS.strip().splitlines():
    print(" ", line)

# Optional live Neo4j check (gracefully falls back to offline snapshot mode)
try:
    with KnowledgeGraph.connect() as live_kg:
        live_rows = live_kg.training_rows()
        print(f"\n[Neo4j LIVE] Connected to bolt://localhost:7687 — {len(live_rows):,} training rows available.")
except Exception as exc:
    print(f"\n[Offline Snapshot Mode] No local Neo4j server active ({type(exc).__name__}); reading exported parquet snapshot.")

In [ ]:
%%pgen
import polars as pl

from src.config import get_settings

pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(38)

kg = get_settings().paths.library / "kg"

allele_variants = pl.read_parquet(kg / "allele_variants.parquet")
annotation_variants = pl.read_parquet(kg / "annotation_variants.parquet")
annotation_edges = pl.read_parquet(kg / "annotation_edges.parquet")

for name, frame in [
    ("allele_variants", allele_variants),
    ("annotation_variants", annotation_variants),
    ("annotation_edges", annotation_edges),
]:
    print(f"{name:22} {frame.height:>7} rows x {frame.width} columns")

### Why a graph database? Multi-hop resolution into canonical GRCh38 SPDI

Clinical pharmacogenetics sources name the same genomic change in three incompatible ways:
- **Star alleles / haplotypes** (`CYP2C19*2`)
- **Current or retired dbSNP rsIDs** (`rs4244285`, or retired `rs17879456` merged into `rs4244285`)
- **Genomic or coding HGVS** (`NC_000010.11:g.94781859G>A`)

In the knowledge graph, `build_kg` (`src/kg/build.py`) resolves all three paths onto a single canonical **GRCh38 SPDI `Variant` node** (`NC_000010.11:94781858:G:A`), while preserving every hop and its dataset provenance (`pharmgkb`, `pharmvar`, `dbsnp`, `ncbi_spdi`). Below we run `build_kg` in-memory on the offline fixture set and draw the clopidogrel / `CYP2C19*2` subgraph.

In [ ]:
%%pgen
import matplotlib.pyplot as plt
import networkx as nx

# 1. Inspect the multi-hop paths in the exported KG tables that converge on NC_000010.11:94781858:G:A
target_spdi = "NC_000010.11:94781858:G:A"
print(f"Allele labels in allele_variants.parquet resolving to canonical SPDI {target_spdi}:")
print(
    allele_variants.filter(pl.col("spdi") == target_spdi)
    .select("label", "kind", "path", "relation", "spdi", "hgvs_g", "sources")
    .head(8)
)

print(f"\nSample clinical annotations implicating {target_spdi} on the forward strand (annotation_variants.parquet):")
print(
    annotation_variants.filter(pl.col("spdi") == target_spdi)
    .select("annotation_id", "spdi", "hgvs_g", "method", "evidence", "sources")
    .head(4)
)

# 2. Draw the multi-hop resolution ego-network for Clopidogrel + CYP2C19*2 / rs4244285
G = nx.DiGraph()
ego_nodes = [
    ("Drug\nCHEMBL1771\n(clopidogrel)", "#1b9e77"),
    ("Annotation\n1447985019\n(Efficacy)", "#d95f02"),
    ("Allele\nCYP2C19*2", "#7570b3"),
    ("Allele\nrs4244285", "#7570b3"),
    ("Haplotype\nCYP2C19*2", "#e7298a"),
    ("RefSNP\nrs4244285", "#66a61e"),
    ("RefSNP\nrs17879456\n(retired)", "#a6761d"),
    ("Variant (SPDI)\nNC_000010.11:94781858:G:A\n(g.94781859G>A)", "#e6ab02"),
    ("Gene\nCYP2C19", "#e41a1c"),
]
for n, _ in ego_nodes:
    G.add_node(n)

ego_edges = [
    ("Annotation\n1447985019\n(Efficacy)", "Drug\nCHEMBL1771\n(clopidogrel)", "FOR_DRUG"),
    ("Annotation\n1447985019\n(Efficacy)", "Allele\nCYP2C19*2", "CONCERNS_ALLELE"),
    ("Annotation\n1447985019\n(Efficacy)", "Allele\nrs4244285", "CONCERNS_ALLELE"),
    ("Annotation\n1447985019\n(Efficacy)", "Variant (SPDI)\nNC_000010.11:94781858:G:A\n(g.94781859G>A)", "IMPLICATES\n(allele='A')"),
    ("Allele\nCYP2C19*2", "Haplotype\nCYP2C19*2", "REFERS_TO"),
    ("Allele\nrs4244285", "RefSNP\nrs4244285", "REFERS_TO"),
    ("RefSNP\nrs17879456\n(retired)", "RefSNP\nrs4244285", "MERGED_INTO"),
    ("Haplotype\nCYP2C19*2", "Variant (SPDI)\nNC_000010.11:94781858:G:A\n(g.94781859G>A)", "HAS_VARIANT\n(pharmvar)"),
    ("RefSNP\nrs4244285", "Variant (SPDI)\nNC_000010.11:94781858:G:A\n(g.94781859G>A)", "HAS_ALLELE\n(dbsnp)"),
    ("Variant (SPDI)\nNC_000010.11:94781858:G:A\n(g.94781859G>A)", "Gene\nCYP2C19", "IN_GENE"),
    ("Haplotype\nCYP2C19*2", "Gene\nCYP2C19", "IN_GENE"),
]
for u, v, rel in ego_edges:
    G.add_edge(u, v, label=rel)

pos = {
    "Drug\nCHEMBL1771\n(clopidogrel)": (-2.2, 0.8),
    "Annotation\n1447985019\n(Efficacy)": (-1.0, 0.2),
    "Allele\nCYP2C19*2": (0.1, 0.9),
    "Allele\nrs4244285": (0.1, -0.5),
    "Haplotype\nCYP2C19*2": (1.3, 0.9),
    "RefSNP\nrs4244285": (1.3, -0.5),
    "RefSNP\nrs17879456\n(retired)": (1.3, -1.45),
    "Variant (SPDI)\nNC_000010.11:94781858:G:A\n(g.94781859G>A)": (2.7, 0.2),
    "Gene\nCYP2C19": (4.0, 0.6),
}

fig, ax = plt.subplots(figsize=(13.5, 5.2))
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=[c for _, c in ego_nodes],
    node_size=2400,
    edgecolors="#222222",
    linewidths=1.2,
)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7.5, font_color="white", font_weight="bold")
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color="#555555",
    width=1.5,
    arrows=True,
    arrowsize=16,
    node_size=2500,
    connectionstyle="arc3,rad=0.06",
)
edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax, rotate=False)
ax.set_title("Knowledge Graph Multi-Hop Resolution: Star Alleles & dbSNP rsIDs -> Canonical GRCh38 SPDI Variant")
ax.axis("off")
plt.tight_layout()
plt.show()

### Resolution, and what refused to resolve

Not every label in the sources is a variant that can be placed on the genome.
The resolution status is kept, not discarded — a label that failed to resolve is
recorded with the reason.

In [ ]:
%%pgen
print(allele_variants.group_by("status").agg(n=pl.len()).sort("n", descending=True))

print("\nWhy things did not resolve:")
print(
    allele_variants.filter(pl.col("status") != "resolved")
    .group_by("reason")
    .agg(n=pl.len())
    .sort("n", descending=True)
    .head(10)
)

print("\nA resolved row that carries a genomic placement, with its provenance:")
resolved = allele_variants.filter(
    (pl.col("status") == "resolved") & pl.col("hgvs_g").is_not_null()
).head(1)
for name, value in zip(resolved.columns, resolved.row(0)):
    print(f"  {name:12} {value}")

Each reason is a distinct, deliberate decision:

- `structural_allele` — gene duplications, deletions and hybrids (`CYP2D6*5`
  and friends). They are real pharmacogenetic alleles but not single base
  changes, so they have no HGVS `g.` notation. This is the same gap noted in
  notebook `01`: the tower cannot yet represent them.
- `not_a_variant` — labels like "ABCB1 intermediate activity", which are
  *outcomes*, not genomic changes, and correctly have no placement.
- `not_in_pharmvar_or_tables` — a haplotype no source defines at variant level.
- `no_grch38_placement` / `rsid_not_found` — genuinely unresolvable; only 5 rows.

Note also that a star allele can be `resolved` as a haplotype and still carry no
variants of its own: `CYP2C19*1` is the reference allele, so it resolves to the
gene with zero changes. Keeping every one of these with its reason, rather than
dropping it, is what lets `audit` prove nothing vanished silently.

### The audit reports

In [ ]:
%%pgen
from pathlib import Path

reports = Path("reports/kg")
for path in sorted(reports.glob("*.tsv")):
    lines = path.read_text().splitlines()
    print(f"\n=== {path}  ({len(lines) - 1} rows)")
    for line in lines[:4]:
        print("   ", line[:170])

### Forward-strand allele inference (`IMPLICATES`) and cross-source disagreement auditing

PharmGKB clinical annotations often report genotypes on the gene's coding strand (which for minus-strand genes like `CYP2D6` or `ABCB1` is the reverse complement of GRCh38), or report palindromic `A/T` and `C/G` transversions where a single genotype group (`GG`) cannot disambiguate strand orientation.

`infer_variant` (`src/kg/allele_inference.py`) compares the annotation's genotype groups (`comparison_alleles`) against the dbSNP GRCh38 alleles on both strands and only creates an `[:IMPLICATES]` edge when the forward-strand allele is unambiguous:

In [ ]:
%%pgen
from src.kg.allele_inference import infer_variant
from src.kg.variant import CanonicalVariant

v_ga = CanonicalVariant.from_spdi("NC_000010.11", 94781858, "G", "A", "NC_000010.11:g.94781859G>A")
v_cg = CanonicalVariant.from_spdi("NC_000007.14", 87509328, "C", "G", "NC_000007.14:g.87509329C>G")
v_ca = CanonicalVariant.from_spdi("NC_000007.14", 87509328, "C", "A", "NC_000007.14:g.87509329C>A")
v_ct = CanonicalVariant.from_spdi("NC_000007.14", 87509328, "C", "T", "NC_000007.14:g.87509329C>T")

cases = [
    ("Forward-strand G>A (AG vs GG)", "AG", "GG", [v_ga]),
    ("Minus-strand reported CT vs CC at G>A", "CT", "CC", [v_ga]),
    ("Palindromic C>G (CG vs CC -> ambiguous)", "CG", "CC", [v_cg]),
    ("Multi-allelic C>A,T (AA vs AT + TT)", "AA", "AT + TT", [v_ca, v_ct]),
]

print("Live demonstration of src.kg.allele_inference.infer_variant():")
for desc, studied, comp, site_vars in cases:
    res = infer_variant(studied, comp, site_vars)
    resolved_hgvs = res.variant.hgvs_g if res.variant is not None else "None"
    print(f"  {desc:42} -> variant={resolved_hgvs:28} reason={str(res.reason):24} ({res.evidence})")

print("\nFull corpus annotation_variants.parquet IMPLICATES method breakdown:")
print(annotation_variants.group_by("method").agg(n=pl.len()).sort("n", descending=True))

res_report = pl.read_csv("reports/kg/resolution_report.tsv", separator="\t")
print("\nResolution report breakdown by kind and path (reports/kg/resolution_report.tsv):")
print(res_report.group_by("kind", "status", "path").agg(n_labels=pl.len(), total_annotations=pl.col("n_annotations").sum()).sort("total_annotations", descending=True))

---

## The training rows

One schema, whatever the source: `src.kg.training.to_training_frame` maps Neo4j
rows onto the same columns the training TSV uses, so a snapshot and a live Neo4j
connection are interchangeable (`--neo4j` versus `-i training_rows.parquet`).

In [ ]:
%%pgen
rows = pl.read_parquet(kg / "training_rows.parquet")
print(f"{rows.height} rows x {rows.width} columns\n")
print(dict(rows.schema))

print("\nThe clopidogrel / CYP2C19 rows — the case from notebook 02, as training data:")
print(
    rows.filter(pl.col("drugs").str.contains("clopidogrel") & (pl.col("gene") == "CYP2C19"))
    .select("drugs_chembl", "gene", "genotype", "phenotype_category", "direction_of_effect", "pmid")
    .head(5)
)

pair = rows.filter(
    (pl.col("drugs_chembl") == "CHEMBL1771")
    & (pl.col("genotype") == "NC_000010.11:g.94781859G>A")
)
print(f"\nThat one drug-variant pair appears in {pair.height} rows, "
      f"from {pair['pmid'].n_unique()} distinct publications.")

That last number is the reason the evaluation protocol exists. The same
drug-variant pair recurs across many papers, so a **random** train/test split
would put near-duplicates of the same fact on both sides and report a score that
says nothing about generalization. Hence the grouped scenarios below.

In [ ]:
%%pgen
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (column, title, n) in zip(
    axes,
    [("phenotype_category", "Phenotype category", 8), ("gene", "Top genes", 12), ("source", "Source", 6)],
):
    counts = rows.group_by(column).agg(n=pl.len()).sort("n", descending=True).head(n)
    ax.barh(counts[column].to_list()[::-1], counts["n"].to_list()[::-1], color="#7570b3")
    ax.set_title(title)
    ax.set_xlabel("rows")

plt.tight_layout()
plt.show()

print("Class rows (a drug class rather than a named drug):",
      rows.filter(pl.col("is_class_row"))["is_class_row"].len(), f"of {rows.height}")

---

## The evaluation protocol

A random split badly overstates performance on this data: the same drug–variant
pair recurs across papers, so a random test set is mostly pairs the model has
already seen. The protocol therefore uses **grouped k-fold** scenarios, each
holding out a different kind of entity:

| Scenario | Held out | Question it answers |
|---|---|---|
| `random` | nothing (rows shuffled) | the optimistic baseline, for reference |
| `pair` | drug–variant pairs | unseen *combinations* of known entities |
| `pmid` | publications | generalization across studies |
| `drug_cold` | drugs | a drug the model never saw |
| `variant_cold` | variants | a variant the model never saw |
| `gene_cold` | genes | a gene the model never saw |

Splits are written to `data/splits/<scenario>/` and bound by sha256 to both the
data file and the HGVS catalog. Rebuilding either invalidates them — a
deliberate fail-fast, so results can never be reported against a split that no
longer matches the data.

In [ ]:
%%pgen
import numpy as np
from src.evaluation.runs import catalog_sha256
from src.evaluation.entities import Scenario, file_sha256
from src.evaluation.splits import load_split, verify_split

data_sha = file_sha256(kg / "training_rows.parquet")
catalog_sha = catalog_sha256()

scenarios = [s.value for s in Scenario]
fold0_summary = {}

print(f"Verifying all {len(scenarios)} split scenarios against data ({data_sha[:12]}) & catalog ({catalog_sha[:12]}):\n")
header = f"{'Scenario':14} {'k':>2} {'Train':>7} {'Val':>6} {'Test':>6} {'Drop':>6}  {'Class drop/fold'}"
print(header)
print("-" * len(header))

for scenario in scenarios:
    assignment, manifest = load_split(scenario)
    verify_split(manifest, data_sha256=data_sha, catalog_sha256=catalog_sha)
    f0 = manifest.rows_per_fold[0]
    fold0_summary[scenario] = f0
    print(
        f"{scenario:14} {manifest.k:>2} {f0.get('train', 0):>7,} {f0.get('val', 0):>6,}"
        f" {f0.get('test', 0):>6,} {f0.get('drop', 0):>6,}  {manifest.class_rows_dropped_per_fold}"
    )

# Plot Fold 0 partition sizes across all 6 scenarios
fig, ax = plt.subplots(figsize=(11, 3.8))
parts = ["train", "val", "test", "drop"]
part_colors = {"train": "#1b9e77", "val": "#7570b3", "test": "#d95f02", "drop": "#bababa"}
left = np.zeros(len(scenarios))
for part in parts:
    vals = np.array([fold0_summary[s].get(part, 0) for s in scenarios])
    ax.barh(scenarios[::-1], vals[::-1], left=left[::-1], label=part, color=part_colors[part])
    left += vals
ax.set_xlabel("rows (out of 26,544)")
ax.set_title("Fold 0 partition breakdown across all 6 evaluation scenarios")
ax.legend(loc="lower right", ncol=4)
plt.tight_layout()
plt.show()

The `dropped` count is the leakage guard at work. In `drug_cold`, a row is
dropped from a fold when keeping it would leak a held-out drug — most often a
class row whose ATC pool contains that drug. Dropping is the honest option: the
alternative is a test score inflated by rows the model saw in another form.

In [ ]:
%%pgen
from src.data.cache import GraphCache
from src.data.genotype import load_hgvs_catalog
from src.evaluation.entities import EntityResolver, add_row_id

assignment, manifest = load_split("drug_cold")
print("Sample assignment table (drug_cold):")
print(assignment.head(5))

# 1. Explicit Zero-Leakage Proof on Fold 0 across grouped scenarios
print("\n=== Zero-Leakage Verification (Fold 0) ===")
catalog = load_hgvs_catalog()
resolver = EntityResolver(catalog, drug_col="drugs_chembl")
rows_with_id = add_row_id(rows)

grouped_scenarios = [
    Scenario.DRUG_COLD,
    Scenario.VARIANT_COLD,
    Scenario.GENE_COLD,
    Scenario.PMID,
    Scenario.PAIR,
]

for sc in grouped_scenarios:
    f_assign, _ = load_split(sc.value)
    # Align assignment by row_id
    joined = rows_with_id.join(f_assign.select("row_id", "fold_0"), on="row_id", how="left")
    parts = joined["fold_0"].to_list()
    row_keys = resolver.keys(joined, sc)

    def collect_set(target_part: str) -> set[str]:
        out: set[str] = set()
        for keys, part in zip(row_keys, parts):
            if part == target_part:
                out.update(k for k in keys if not k.startswith("ATC:"))
        return out

    tr_set = collect_set("train")
    val_set = collect_set("val")
    te_set = collect_set("test")
    overlap_tr_te = tr_set & te_set
    overlap_val_te = val_set & te_set
    assert len(overlap_tr_te) == 0 and len(overlap_val_te) == 0
    print(f"  {sc.value:14}: |train|={len(tr_set):5d}, |val|={len(val_set):5d}, |test|={len(te_set):5d}  ->  train∩test = {len(overlap_tr_te)}, val∩test = {len(overlap_val_te)}  ✓")

# 2. Pool-level ATC class defense: GraphCache.without_members(held_out_drugs)
cache = GraphCache.from_settings(inference_mode=True)
dc_joined = rows_with_id.join(assignment.select("row_id", "fold_0"), on="row_id", how="left")
dc_keys = resolver.keys(dc_joined, Scenario.DRUG_COLD)
held_out_drugs = {
    k
    for keys, part in zip(dc_keys, dc_joined["fold_0"].to_list())
    if part in ("val", "test")
    for k in keys
    if not k.startswith("ATC:")
}
trimmed_cache = cache.without_members(held_out_drugs)

print(f"\n=== Pool-level ATC Class Defense in drug_cold Fold 0 ({len(held_out_drugs)} held-out drugs) ===")
for atc_key in ("ATC:C01", "ATC:C10AA", "ATC:N02BE"):
    _, full_mols = cache.get_entity(atc_key)
    _, safe_mols = trimmed_cache.get_entity(atc_key)
    removed = sorted({m.drug_id for m in full_mols} - {m.drug_id for m in safe_mols})
    print(f"  {atc_key:10}: full pool = {len(full_mols):2d} drugs -> fold-0 safe pool = {len(safe_mols):2d} drugs (stripped {len(removed)} held-out drugs: {removed[:4]})")

---

## Metrics from a previous run

The run below was evaluated on an **earlier data snapshot** — its `data_sha256`
does not match the current `training_rows.parquet`, because the library was
rebuilt since. The numbers are shown to demonstrate the measurement, not as the
project's current performance; a model trained on the present library has not
been evaluated yet.

In [ ]:
%%pgen
import json
from pathlib import Path

runs = sorted(Path("reports/eval").iterdir())
print("Available runs:", [r.name for r in runs], "\n")

metrics = json.loads((runs[-1] / "metrics.json").read_text())

print(f"model     {metrics['model']}")
print(f"scenario  {metrics['scenario']}  (k={metrics['k']}, folds evaluated: {metrics['folds_evaluated']})")
print(f"test rows {metrics['n_test']}   bootstrap resamples: {metrics['n_bootstrap']}")
print(f"data      {metrics['data_sha256'][:12]}  <- current data is {data_sha[:12]}")
print(f"           {'MATCHES' if metrics['data_sha256'] == data_sha else 'STALE: predates the current library'}")

HEADLINE = (
    "f1_micro", "f1_macro", "f1_weighted", "precision_macro", "recall_macro",
    "roc_auc_macro", "pr_auc_macro", "subset_accuracy", "hamming_loss", "ece",
)
print(f"\nAggregate, bootstrap 95% confidence intervals "
      f"({len(metrics['aggregate'])} metrics computed, {len(HEADLINE)} shown):")
for name in HEADLINE:
    ci = metrics["aggregate"][name]
    print(f"  {name:18} {ci['value']:.3f}  [{ci['low']:.3f}, {ci['high']:.3f}]")

In [ ]:
%%pgen
header = f"{'label':16} {'support':>7} {'prev':>6}  {'F1':>5} {'precis':>7} {'recall':>7}  reliable"
print(header)
print("-" * len(header))
for entry in metrics["per_class"]:
    print(
        f"{entry['label']:16} {entry['support']:>7} {entry['prevalence']:>6.2f}"
        f"  {entry['f1']['value']:>5.3f} {entry['precision']['value']:>7.3f}"
        f" {entry['recall']['value']:>7.3f}  {entry['reliable']}"
    )

print("\nBaselines — what a trivial model scores on the same split:")
for baseline in metrics["baselines"]:
    ci = baseline["aggregate"]["f1_macro"]
    print(f"  {baseline['name']:14} f1_macro {ci['value']:.3f}  [{ci['low']:.3f}, {ci['high']:.3f}]")
model_ci = metrics["aggregate"]["f1_macro"]
print(f"  {'the model':14} f1_macro {model_ci['value']:.3f}"
      f"  [{model_ci['low']:.3f}, {model_ci['high']:.3f}]")

This is why the baselines are reported alongside the model. Read honestly:

- The model's macro-F1 (0.557) is above the strongest trivial baseline,
  `gene_only` (0.503) — but the model's 95% interval runs down to 0.377, *below*
  that baseline's point estimate. On one fold of `drug_cold`, the model is *not*
  convincingly better than predicting from the gene alone. Five folds and a
  retrained model are needed before claiming otherwise.
- `gene_only` scoring 0.503 is itself the finding: most of the signal in this
  data is carried by which gene is involved, not by the drug. A model that beats
  it has learned something about the drug-genotype interaction.
- `PD` has a support of 3 rows and is flagged `reliable: False`. Its 0.000 F1 is
  not a result, it is an absence of test data, which is why the protocol marks
  such labels instead of averaging them into a headline number.

Reporting the interval and the baseline, rather than the point estimate alone,
is the protocol's main defence against over-claiming.

Each run also writes a self-contained `report.html` rendering every field of
`metrics.json`:

In [ ]:
%%pgen
for run in runs:
    contents = sorted(p.name for p in run.iterdir())
    print(f"{run.name}\n  {contents}\n")

---

## Training: the placeholder

No model is trained in these notebooks. The library was rebuilt, so every
checkpoint has to be retrained from scratch, and a real run needs a GPU and
hours rather than a demo cell.

The commands below are the whole interface. Run on a GPU Colab runtime, the
first one trains one fold of one scenario.

In [ ]:
# Not executed here. This is the full training and evaluation interface.
#
# One fold of one scenario (artifacts land under run_name(model, FoldRef)):
#   python main.py train -i data/library/kg/training_rows.parquet \
#       --scenario drug_cold --fold 0 -M TwoTowerGAT -e 100
#
# All five folds, then aggregate them into a report:
#   for k in 0 1 2 3 4; do
#       python main.py train -i data/library/kg/training_rows.parquet \
#           --scenario drug_cold --fold $k
#   done
#   python main.py evaluate -i data/library/kg/training_rows.parquet \
#       --scenario drug_cold --bootstrap 2000
#
# Hyperparameter search:
#   python main.py optuna --trials 50 --epochs 30 --sampler tpe
#
# Compare two runs, or re-render a report:
#   python main.py evaluate --compare reports/eval/<A> reports/eval/<B>
#   python main.py evaluate --render reports/eval/<A>
print("Reference only — see the markdown above.")

To confirm the training path is wired correctly without a GPU, the integration
test trains for a few steps on synthetic graphs and then predicts:

```bash
uv run pytest tests/integration/test_train_predict_e2e.py -q
```

---

## Summary

- The knowledge graph is the single source of training rows, with mandatory
  provenance on every fact and an audit that proves it.
- Variants are canonical GRCh38 SPDI; labels that are not variants are kept with
  a reason rather than dropped.
- Evaluation is grouped k-fold across six scenarios, bound to the data by
  sha256, with a leakage guard that drops rows rather than inflate a score.
- Metrics are reported as bootstrap intervals against trivial baselines.
- Training is the one piece awaiting a run on the current library.